# Дообучение MobileNetV2 на PyTorch (Mac-friendly)

Работает на Mac из коробки, включая ускорение через Metal (MPS) на Apple Silicon.

Перед запуском: соберите датасет через `collect_dataset.py` и разбейте через `split_dataset.py`,
чтобы получить `dataset_split/train/<class>/*.jpg` и `dataset_split/val/<class>/*.jpg`.

In [ ]:
# Установка (один раз)
# !pip install torch torchvision pillow

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from pathlib import Path
import time

# Выбор устройства: MPS (Apple GPU) -> CUDA -> CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Устройство:", device)

In [ ]:
DATA_DIR = Path("./dataset_split")
IMG_SIZE = 224
BATCH_SIZE = 32

# ImageNet-нормализация — MobileNetV2 предобучен на ней
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_ds = datasets.ImageFolder(DATA_DIR / "val", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_names = train_ds.classes
num_classes = len(class_names)
print("Классы:", class_names)
print("Train:", len(train_ds), " Val:", len(val_ds))

In [ ]:
# Модель: предобученный MobileNetV2, своя голова классификатора
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Замораживаем всю базовую сеть на фазе 1
for param in model.features.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, num_classes),
)

model = model.to(device)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


def train_loop(model, epochs, lr, params_to_optimize=None):
    criterion = nn.CrossEntropyLoss()
    params = params_to_optimize if params_to_optimize is not None else model.parameters()
    optimizer = torch.optim.Adam(params, lr=lr)

    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        dt = time.time() - t0
        print(f"Эпоха {epoch+1}/{epochs} ({dt:.1f}с)  "
              f"train_loss={train_loss:.3f} train_acc={train_acc:.3f}  "
              f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}")

In [ ]:
print("=== Фаза 1: обучение головы классификатора (база заморожена) ===")
train_loop(model, epochs=15, lr=1e-3, params_to_optimize=model.classifier.parameters())

In [ ]:
print("=== Фаза 2: fine-tuning последних слоёв базовой сети ===")

# Размораживаем последние ~30% слоёв features
feature_layers = list(model.features.children())
unfreeze_from = int(len(feature_layers) * 0.7)
for layer in feature_layers[unfreeze_from:]:
    for param in layer.parameters():
        param.requires_grad = True

train_loop(model, epochs=10, lr=1e-5, params_to_optimize=filter(lambda p: p.requires_grad, model.parameters()))

In [ ]:
# Сохранение модели и списка классов
torch.save(model.state_dict(), "surface_mobilenet.pt")
with open("surface_mobilenet.classes.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))
print("Сохранено: surface_mobilenet.pt, surface_mobilenet.classes.txt")

## Экспорт для мобильного инференса

Если планируете использовать модель в мобильном приложении:
- **iOS** → CoreML через `coremltools`
- **Android** → TFLite (через ONNX) или PyTorch Mobile (`.ptl` через `torch.jit`)

Пример экспорта в CoreML ниже (нужно `pip install coremltools`).

In [ ]:
# import coremltools as ct
#
# model.eval()
# example_input = torch.rand(1, 3, IMG_SIZE, IMG_SIZE).to(device)
# traced_model = torch.jit.trace(model.cpu(), example_input.cpu())
#
# mlmodel = ct.convert(
#     traced_model,
#     inputs=[ct.ImageType(name="input", shape=example_input.shape, scale=1/255.0)],
#     classifier_config=ct.ClassifierConfig(class_names),
# )
# mlmodel.save("SurfaceClassifier.mlmodel")